# OCR Extraction Pipeline — Setup & Test

End-to-end extraction pipeline for university grant administration documents using a Vision Language Model (VLM).

**What this notebook does:**
1. Loads a VLM (Qwen3-VL-32B or Qwen2.5-VL-72B) onto GPU
2. Renders PDF pages as images and sends each to the VLM for structured extraction
3. Extracts: narratives (verbatim), tables, stakeholders, addresses, signatures, annotations
4. Assembles per-page results into a document-level JSONL matching the grant admin schema
5. Batch processes all PDFs in `/ocr/`
6. Compares VLM output against Gemini reference JSONs with plots

**Before starting:** Upload your sample PDFs to `/ocr/` and Gemini reference JSONs to `/ocr/gemini/` using Jupyter's file upload button.


In [ ]:
from pathlib import Path
import os

# ── VLM mode: "remote" (shared vLLM endpoint) or "local" (load on this GPU) ──
VLM_MODE = "remote"

# Remote mode settings (vLLM endpoint)
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://qwen3--vl--32b--instruct-awq.runai-shared-models.svc.cluster.local/v1")
VLLM_MAX_TOKENS = 8192

# Model name — used for output metadata. In remote mode, auto-detected from endpoint.
MODEL_NAME = os.environ.get("VLM_MODEL", "QuantTrio/Qwen3-VL-32B-Instruct-AWQ")

if VLM_MODE == "remote":
    import httpx
    # Auto-detect model name from vLLM endpoint
    try:
        resp = httpx.get(f"{VLLM_BASE_URL}/models", timeout=5)
        resp.raise_for_status()
        models = resp.json().get("data", [])
        if models:
            MODEL_NAME = models[0]["id"]
        print(f"Remote VLM endpoint: {VLLM_BASE_URL}")
        print(f"Model: {MODEL_NAME}")
        print("Status: CONNECTED")
    except Exception as e:
        print(f"Remote VLM endpoint: {VLLM_BASE_URL}")
        print(f"Status: UNREACHABLE ({e})")
        print("\nFix: update VLLM_BASE_URL above, or switch to VLM_MODE = \"local\"")
else:
    # Local mode: set HF cache paths for model loading
    os.environ["HF_HOME"] = "/models/.cache/huggingface"
    os.environ["HF_HUB_CACHE"] = "/models/.cache/huggingface"
    os.environ["HF_HUB_OFFLINE"] = "1"

    models_dir = Path("/models/.cache/huggingface")
    if models_dir.exists():
        model_dirs = [d.name for d in models_dir.iterdir() if d.name.startswith("models--")]
        print(f"Local mode. Shared models PVC: {len(model_dirs)} model(s)")
        has_vlm = any("Qwen3-VL" in d or "Qwen2.5-VL" in d for d in model_dirs)
        if not has_vlm:
            print("WARNING: No Qwen VL model found!")
    else:
        print("WARNING: /models/.cache/huggingface not found.")


### GPU check (local mode only)

Runs `nvidia-smi` when `VLM_MODE = "local"` so you can confirm the workspace has a GPU and see its memory. Silently skipped in remote mode since the workspace is CPU-only.


In [ ]:
if VLM_MODE == "local":
    import subprocess
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free",
             "--format=csv,noheader"],
            text=True, stderr=subprocess.STDOUT)
        print(out)
    except (subprocess.CalledProcessError, FileNotFoundError) as e:
        print(f"nvidia-smi not available: {e}")
        print("Local mode requires GPU access — check workspace compute settings.")
else:
    print("Remote mode — GPU check skipped (vLLM server has the GPU, workspace is CPU-only)")

## 2. Load the model (local mode only)

**Skip this cell if using `VLM_MODE = "remote"`** — the model is already loaded on the vLLM server.

### Requirements for local mode

If you want to run the model directly in the notebook, the workspace needs:

1. **GPU fraction on the workspace itself** — edit the workspace in RunAI UI:
   - `25%` of a 96 GB GPU for the AWQ 4-bit model (~20 GB VRAM)
   - `75-85%` for the full bf16 model (~64 GB VRAM)

2. **Shared models PVC mounted** at `/models` — add `shared-models` data volume in the workspace config

3. **HF env vars** set on the workspace:
   - `HF_HOME=/models/.cache/huggingface`
   - `HF_HUB_CACHE=/models/.cache/huggingface`
   - `HF_HUB_OFFLINE=1` (prevents accidental downloads)

4. **Model weights already on the PVC.** Before running the load cell, verify:

   ```bash
   ls /models/.cache/huggingface/ | grep Qwen
   ```

   You should see a directory matching your chosen `MODEL_NAME`. If not, the
   model needs to be downloaded first — see [Setup Data Volumes](../docs/setup-data-volumes.md)
   for download instructions (run from the `shared-models` project).

5. **`transformers` + `qwen-vl-utils` installed** — the install cell below handles this
   automatically when `VLM_MODEL = "local"`.

Change `MODEL_NAME` below to switch models. The `Auto` class handles both Qwen2.5-VL and Qwen3-VL architectures automatically.


In [ ]:
# Skip this cell in remote mode — model is on the vLLM server
if VLM_MODE == "local":
    import time
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

    # ── Choose a model (uncomment one) ─────────────────────────────────
    # MODEL_NAME = "Qwen/Qwen3-VL-32B-Instruct"          # ~64 GB bf16
    MODEL_NAME = "QuantTrio/Qwen3-VL-32B-Instruct-AWQ"  # ~20 GB INT4 — default
    # MODEL_NAME = "Qwen/Qwen2.5-VL-72B-Instruct-AWQ"  # ~40 GB INT4
    # MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"          # ~16 GB bf16

    USE_BNB_4BIT = False
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    ) if USE_BNB_4BIT else None

    print(f"Loading {MODEL_NAME}...")
    t0 = time.time()
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="flash_attention_2",
        quantization_config=bnb_config,
    )
    elapsed = time.time() - t0
    print(f"Model loaded in {elapsed:.1f}s — Device: {model.device}")

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            total = torch.cuda.get_device_properties(i).total_memory / 1024**3
            used = torch.cuda.memory_allocated(i) / 1024**3
            print(f"GPU {i}: {used:.1f} / {total:.1f} GB")
else:
    print("Remote mode — skipping local model load.")
    print(f"Using vLLM endpoint: {VLLM_BASE_URL}")

    # Smoke test: send a trivial request to confirm the endpoint responds
    import time, httpx
    print("\nRunning smoke test...")
    t0 = time.time()
    try:
        resp = httpx.post(
            f"{VLLM_BASE_URL}/chat/completions",
            json={
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": "Say OK."}],
                "max_tokens": 10,
                "temperature": 0.0,
            },
            timeout=60,
        )
        resp.raise_for_status()
        reply = resp.json()["choices"][0]["message"]["content"].strip()
        elapsed = time.time() - t0
        print(f"  Reply: {reply!r}")
        print(f"  Elapsed: {elapsed:.1f}s")
        print(f"  Smoke test PASSED — endpoint is live and generating.")
    except Exception as e:
        print(f"  Smoke test FAILED after {time.time()-t0:.1f}s: {e}")
        print(f"  Check that the vLLM job is Running at {VLLM_BASE_URL}")


### Install local-mode dependencies (local mode only)

Skip this cell in remote mode — all base packages are already installed by the workspace startup.

For local mode, this installs `transformers`, `qwen-vl-utils`, and `autoawq` if not already present.


In [ ]:
# Local mode needs extra packages (transformers, qwen-vl-utils, autoawq)
# beyond what the workspace startup installs. Skip this cell in remote mode.

if VLM_MODE == "local":
    import importlib.util
    local_deps = ["transformers", "qwen_vl_utils", "autoawq"]
    missing = [p for p in local_deps if importlib.util.find_spec(p) is None]
    if missing:
        print(f"Installing local mode deps: {missing}")
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "--break-system-packages", "-q"] + missing)
        print("Done.")
    else:
        print("Local mode packages OK: transformers, qwen-vl-utils, autoawq")
else:
    print("Remote mode — nothing to install here.")


### Define helper functions

- `run_vlm(messages)` — core inference: dispatches to remote vLLM endpoint or local transformers model based on `VLM_MODE`
- `extract_page(image, prompt)` — encodes a page image as base64 PNG and sends it with the extraction prompt to the VLM. Supports sliding window context (prev/next page images).
- `extract_links(page)` — extracts hyperlinks from a PDF page's metadata layer

These functions work identically in both remote and local mode — the only difference is where the model runs.


In [ ]:
import base64, io, time

def _encode_image_b64(image):
    """Encode a PIL Image as a base64 PNG data URI."""
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"

def _run_vlm_remote(messages, max_tokens=16000):
    """Send messages to the vLLM OpenAI-compatible API."""
    import httpx

    # Convert messages to OpenAI format (base64 images stay as-is)
    oai_messages = []
    for msg in messages:
        oai_content = []
        for part in msg["content"]:
            if part["type"] == "text":
                oai_content.append({"type": "text", "text": part["text"]})
            elif part["type"] == "image":
                image_url = part["image"]
                oai_content.append({"type": "image_url", "image_url": {"url": image_url}})
        oai_messages.append({"role": msg["role"], "content": oai_content})

    payload = {
        "model": MODEL_NAME,
        "messages": oai_messages,
        "max_tokens": max_tokens,
        "temperature": 0.0,
        # NOTE: do NOT set frequency_penalty / repetition_penalty here.
        # Grant docs contain long Standard_Tables whose rows repeat the
        # same JSON keys (e.g. "Report Type", "Frequency", "Copies");
        # token-level repetition penalties cause the VLM to stop emitting
        # rows after only a handful, truncating the tables array. Seen as
        # ~25 tables collapsing to 1 in _extracted.json / _final.json.
        # The library notebook uses these penalties to break decorative
        # caption loops; that use case does not apply to grant admin.
        # Force valid JSON output at the token level. vLLM enforces this
        # during generation so we do not get mixed quote styles, JS-style
        # comments, or trailing garbage.
        "response_format": {"type": "json_object"},
    }
    # Retry on gateway/transient errors — 2 attempts with a 30s pause.
    # Don't hammer — if the backend is saturated, more retries at the same
    # quality level won't help. After 2 × 504, the caller falls back to a
    # lower-quality tier (single-page / lower DPI) instead.
    import time as _time
    _BACKOFF = [0, 30]   # immediate, then one retry after 30s
    _ATTEMPTS = len(_BACKOFF)
    last_exc = None
    for attempt, delay in enumerate(_BACKOFF):
        if delay:
            _time.sleep(delay)
        try:
            resp = httpx.post(
                f"{VLLM_BASE_URL}/chat/completions",
                json=payload,
                timeout=300,  # 5 min — large pages can be slow
            )
            if resp.status_code in (502, 503, 504):
                print(f"    gateway {resp.status_code} on attempt {attempt + 1}/{_ATTEMPTS} — retrying")
                last_exc = httpx.HTTPStatusError(
                    f"Gateway {resp.status_code}", request=resp.request, response=resp
                )
                continue
            resp.raise_for_status()
            break
        except (httpx.TimeoutException, httpx.ReadTimeout, httpx.ConnectError) as e:
            print(f"    network error on attempt {attempt + 1}/{_ATTEMPTS}: {type(e).__name__} — retrying")
            last_exc = e
            continue
    else:
        raise last_exc
    choice = resp.json()["choices"][0]
    if choice.get("finish_reason") == "length":
        print(f"    WARNING: VLM output hit max_tokens ({max_tokens}) — response likely truncated. "
              f"Tables/narratives near the tail may be dropped by parse recovery.")
    return choice["message"]["content"]

def _run_vlm_local(messages, max_tokens=16000):
    """Run inference on the locally loaded model."""
    from qwen_vl_utils import process_vision_info

    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0]

def run_vlm(messages, max_tokens=16000):
    """Run VLM inference — dispatches to remote (vLLM) or local (transformers)."""
    if VLM_MODE == "remote":
        return _run_vlm_remote(messages, max_tokens)
    else:
        return _run_vlm_local(messages, max_tokens)

def extract_links(page):
    """Extract hyperlinks from a PDF page. Returns list of {text, url}."""
    links = []
    for link in page.get_links():
        uri = link.get("uri", "")
        if not uri:
            continue
        rect = link.get("from", None)
        text = page.get_text("text", clip=rect).strip() if rect else ""
        links.append({"text": text, "url": uri})
    return links


def _downscale_context_image(image, max_pixels=800*600):
    """Downscale a context image to reduce token cost.

    Context pages (prev/next in the sliding window) don't need full resolution
    since the VLM only uses them for boundary detection. Downscaling them
    cuts input tokens by ~60% without losing cross-page context.
    """
    w, h = image.size
    if w * h <= max_pixels:
        return image
    scale = (max_pixels / (w * h)) ** 0.5
    new_w, new_h = int(w * scale), int(h * scale)
    return image.resize((new_w, new_h), Image.LANCZOS)


def _is_landscape(image):
    """Check if an image is landscape orientation."""
    return image.size[0] > image.size[1]


def _orientation_differs(img_a, img_b):
    """Check if two images have different orientations (portrait vs landscape)."""
    if img_a is None or img_b is None:
        return False
    return _is_landscape(img_a) != _is_landscape(img_b)

def extract_page(image, prompt, max_tokens=16000, filename=None, links=None,
                 prev_image=None, next_image=None):
    """Send a rendered page image to the VLM for structured extraction.

    Sliding window: if prev_image / next_image are provided, they are
    included as visual context so the VLM can detect cross-page content.
    The VLM extracts data from the CENTER page only.

    Orientation-aware: when context pages have a different orientation
    (portrait vs landscape) from the center page, they are dropped to
    avoid confusing the VLM with mixed-geometry inputs. Context images
    are also downscaled to reduce input token cost.
    """
    # Drop context images that have a different orientation — mixed portrait/
    # landscape windows confuse the VLM's layout understanding and are a
    # leading cause of extraction failures on orientation-transition pages.
    effective_prev = prev_image
    effective_next = next_image
    if _orientation_differs(image, prev_image):
        effective_prev = None
    if _orientation_differs(image, next_image):
        effective_next = None

    # Downscale surviving context images to cut input tokens
    if effective_prev is not None:
        effective_prev = _downscale_context_image(effective_prev)
    if effective_next is not None:
        effective_next = _downscale_context_image(effective_next)

    context_parts = []
    if filename:
        context_parts.append(f"Source filename: {filename}")
    if links:
        link_lines = "\n".join(f"  {l['text']} -> {l['url']}" for l in links)
        context_parts.append(f"Hyperlinks found in this page:\n{link_lines}")
    if effective_prev or effective_next:
        context_parts.append(
            "CONTEXT: Surrounding pages are shown for context only. "
            "Extract data from the CENTER PAGE only. Use the surrounding "
            "pages to determine if content continues across page boundaries "
            "(set continuation flags accordingly)."
        )
    if _is_landscape(image):
        context_parts.append(
            "NOTE: This page is in LANDSCAPE orientation. Pay special attention "
            "to wide tables and ensure all columns and rows are fully captured."
        )
    prefix = "\n\n".join(context_parts)
    text = f"{prefix}\n\n{prompt}" if prefix else prompt

    content = []
    if effective_prev:
        content.append({"type": "text", "text": "[PREVIOUS PAGE — context only]"})
        content.append({"type": "image", "image": _encode_image_b64(effective_prev)})
    content.append({"type": "text", "text": "[CENTER PAGE — extract this page]"})
    content.append({"type": "image", "image": _encode_image_b64(image)})
    if effective_next:
        content.append({"type": "text", "text": "[NEXT PAGE — context only]"})
        content.append({"type": "image", "image": _encode_image_b64(effective_next)})
    content.append({"type": "text", "text": text})

    messages = [{"role": "user", "content": content}]
    return run_vlm(messages, max_tokens)


# Character-level JSON cleanup for VLM responses
_CURLY_QUOTES = {
    "“": "\"", "”": "\"",   # left/right double curly quotes
    "‘": "'",  "’": "'",     # left/right single curly quotes
    "„": "\"", "‟": "\"",   # double low-9 / double high-reversed-9 quotes
    "‹": "'",  "›": "'",     # single angle quotes
    "«": "\"", "»": "\"",   # guillemets (rare in VLM output)
}


def _normalize_vlm_json(text: str) -> str:
    """Normalize common quote/formatting quirks before JSON parsing."""
    # Replace curly/smart quotes with straight quotes
    for bad, good in _CURLY_QUOTES.items():
        text = text.replace(bad, good)
    # Strip JS-style // line comments (VLMs sometimes add these)
    import re
    text = re.sub(r"(?m)\s*//[^\n]*$", "", text)
    # Fix escaped quotes at positions where unescaped are expected.
    # This is a common VLM slip: backslash-escaping every quote even outside
    # strings. Replace \" → " when it appears right after a comma/brace/
    # newline (positions where JSON expects a key/value start).
    text = re.sub(r'([,\{\[\n\r\s])\\"', r'\1"', text)
    return text


def parse_vlm_json(raw: str):
    """Robustly parse a VLM response as JSON.

    Handles common issues:
    - Markdown fences (```json ... ``` or ``` ... ```)
    - Leading/trailing prose
    - Truncated output (fallback to extract {...} substring)
    - Runaway repetition (VLM stuck in a loop) — truncate at repetition,
      rebuild JSON from last balanced brace

    Returns (parsed_dict, error_msg_or_None).
    """
    if not raw or not raw.strip():
        return None, "empty response"

    cleaned = raw.strip()

    # Strip markdown fences
    if cleaned.startswith("```"):
        first_newline = cleaned.find("\n")
        if first_newline != -1:
            cleaned = cleaned[first_newline + 1:]
        cleaned = cleaned.rsplit("```", 1)[0].strip()

    # Normalize common VLM quirks (curly quotes, JS comments, escaped quotes)
    cleaned = _normalize_vlm_json(cleaned)

    # Try direct parse
    first_err_msg = None
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        first_err_msg = str(e)

    # Fallback 1a: blanket-unescape any remaining \" -> ".
    # If the VLM started escaping quotes mid-response (e.g. opens with
    # "key": "value", then shifts to \"key\": \"value\"), this recovers.
    # Only attempt if the string CONTAINS escaped quotes at all.
    if '\\"' in cleaned:
        unescaped = cleaned.replace('\\"', '"')
        try:
            return json.loads(unescaped), None
        except json.JSONDecodeError:
            # Also try on the first-brace-to-last-brace slice
            fb = unescaped.find("{")
            lb = unescaped.rfind("}")
            if fb != -1 and lb > fb:
                try:
                    return json.loads(unescaped[fb:lb + 1]), None
                except json.JSONDecodeError:
                    pass

    # Fallback 1: first { ... last }
    first_brace = cleaned.find("{")
    last_brace = cleaned.rfind("}")
    if first_brace != -1 and last_brace > first_brace:
        try:
            return json.loads(cleaned[first_brace:last_brace + 1]), None
        except json.JSONDecodeError:
            pass

    # Fallback 1b: per-field recovery.
    # If a single field in the middle is malformed but other fields are valid,
    # parse field-by-field and keep only the good ones. Drops just the bad
    # field instead of everything after it.
    def _recover_by_field(text):
        for candidate_text in (text, text.replace('\\"', '"')):
            fb = candidate_text.find("{")
            lb = candidate_text.rfind("}")
            if fb == -1 or lb <= fb:
                continue
            body = candidate_text[fb + 1:lb]

            # Split body into field segments at depth-0 commas (relative to body)
            segments = []
            depth = 0
            in_string = False
            escape = False
            last_start = 0
            for idx, ch in enumerate(body):
                if escape:
                    escape = False
                    continue
                if ch == "\\":
                    escape = True
                    continue
                if ch == '"':
                    in_string = not in_string
                    continue
                if in_string:
                    continue
                if ch in "{[":
                    depth += 1
                elif ch in "}]":
                    depth -= 1
                elif ch == "," and depth == 0:
                    segments.append(body[last_start:idx])
                    last_start = idx + 1
            segments.append(body[last_start:])

            # Test each segment; keep parseable ones
            valid_segments = []
            for seg in segments:
                seg = seg.strip()
                if not seg:
                    continue
                try:
                    json.loads("{" + seg + "}")
                    valid_segments.append(seg)
                except json.JSONDecodeError:
                    continue

            if not valid_segments:
                continue
            reconstructed = "{" + ",".join(valid_segments) + "}"
            try:
                return json.loads(reconstructed)
            except json.JSONDecodeError:
                continue
        return None

    field_recovered = _recover_by_field(cleaned)
    if field_recovered is not None:
        return field_recovered, None

    # Fallback 1c: truncate busted tail.
    # VLMs often produce valid JSON for all the content fields, then garble
    # the last 1-2 fields (continuation, other_metadata). Walk back from
    # the end looking for commas where we can close the JSON with "}" and
    # get a valid parse. Drops the garbled tail fields — they are usually
    # non-critical (continuation is backed up by programmatic detection).
    def _try_truncate_tail(text):
        # Also try on the unescaped version
        for candidate_text in (text, text.replace('\\"', '"')):
            fb = candidate_text.find("{")
            if fb == -1:
                continue
            body = candidate_text[fb:]
            # Find all comma positions at depth 1 (outermost object)
            depth = 0
            in_string = False
            escape = False
            comma_positions = []
            for idx, ch in enumerate(body):
                if escape:
                    escape = False
                    continue
                if ch == "\\":
                    escape = True
                    continue
                if ch == '"' and not escape:
                    in_string = not in_string
                    continue
                if in_string:
                    continue
                if ch == "{" or ch == "[":
                    depth += 1
                elif ch == "}" or ch == "]":
                    depth -= 1
                elif ch == "," and depth == 1:
                    comma_positions.append(idx)
            # Try truncating at each comma position from latest to earliest
            for pos in reversed(comma_positions):
                candidate = body[:pos] + "}"
                try:
                    return json.loads(candidate)
                except json.JSONDecodeError:
                    continue
        return None

    truncated = _try_truncate_tail(cleaned)
    if truncated is not None:
        return truncated, None

    # Fallback 2: detect runaway repetition and recover from last valid balanced position.
    # VLM repetition looks like the same short phrase appearing 10+ times — truncate at
    # the start of the repetition, then walk backwards to find a balanced JSON prefix.
    import re
    rep_match = re.search(r"(.{5,50}?)\1{4,}", cleaned)
    if rep_match:
        trunc_at = rep_match.start()
        trimmed = cleaned[:trunc_at]
        for i in range(len(trimmed), max(0, len(trimmed) - 5000), -1):
            candidate = trimmed[:i].rstrip(",\n \t")
            for suffix in ('"}', '"}]}', '"]}', '}', ']}', '"}}', '""}]}'):
                try:
                    return json.loads(candidate + suffix), None
                except json.JSONDecodeError:
                    continue
        return None, f"runaway repetition detected but recovery failed"

    return None, f"invalid JSON: {first_err_msg}"


print(f"Helper functions ready (VLM_MODE={VLM_MODE}).")


### Extraction prompt, filename parser, and assembly function

This cell defines three things:

1. **`EXTRACTION_PROMPT`** — the per-page prompt sent to the VLM. Based on the grant admin spec, it asks for: narratives (verbatim with `[cite: N]`), tables (3 classification types), stakeholders, addresses, signatures, annotations, watermarks, confidence, and document details.

2. **`parse_filename()`** — parses structured filenames like `A_RSP Public_AWD-001064_3718296725__AWD00000002__A_RSP_Award_Notice_of_Award.pdf` into metadata fields (Drawer, AwardID, Field2-5, DocumentType). Double underscores (`__`) indicate empty fields.

3. **`assemble_document_jsonl()`** — merges per-page VLM results into the final document-level JSON matching the spec. Maps snake_case VLM output to PascalCase schema, aggregates across pages.


### Per-page extraction prompt (editable in its own cell)

Edit `EXTRACTION_PROMPT` below to tweak what the VLM is asked to extract per page. Re-run this cell after changes — the batch loop uses `EXTRACTION_PROMPT` directly.


In [ ]:
EXTRACTION_PROMPT = """Role & Objective: You are an expert data extraction assistant specializing in university grant administration documents. Data quality is paramount: do not abbreviate, shorten, or use external assumptions to fill in missing values.

Task: Extract all data from this single document page into the JSON structure below. Do not analyze data.

{
  "confidence_percentage": <float 0-100>,
  "confidence_narrative": "<brief note on extraction quality and comprehensiveness>",
  "has_annotation": <boolean>,
  "has_watermark": <boolean>,
  "signature_lines": {
    "has_signature_line": <boolean>,
    "has_valid_signature": <boolean>
  },
  "document_tags": ["<high-level grant admin tags, e.g. IRB, IACUC, Biosafety>"],
  "one_sentence_summary": "<one sentence summary>",
  "document_details": {
    "application_id": "", "application_type": "", "title": "",
    "requested_amount": null, "completed_date": "", "sub_document_type": ""
  },
  "stakeholders": [
    {
      "context_snippet": "<3-5 words near the stakeholder info>",
      "stakeholder_role": "<Principal Investigator | Co-Investigator | Collaborator | Key Personnel | Grants Administrative Contact | Sponsor Contact | Authorized Organizational Representative | Unknown>",
      "full_name": "", "first_name": "", "last_name": "",
      "email": "", "phone": "", "institution": "", "department": "",
      "position_title": "", "highest_education": "",
      "raw_stakeholder_text": "<verbatim text block containing stakeholder info>"
    }
  ],
  "addresses": [
    {
      "context_snippet": "<3-5 words near the address>",
      "addressee": "", "care_of": null,
      "address_line1": "", "address_line2": "",
      "city": "", "state_province": "", "postal_code": "", "country": "",
      "stakeholder_type": "<Funding Agency | Grantee Institution | Subrecipient | Principal Investigator | Grants Administrative Contact | Unknown>",
      "raw_address_text": "<verbatim text block containing the full address>"
    }
  ],
  "tables": [
    {
      "table_classification": "<Literal_Grid | Key_Value_Form | Standard_Table>",
      "table_data": "<see classification rules below>"
    }
  ],
  "narrative_responses": [
    {
      "prompt_or_header": "<exact question, section header, or 'General Body Text'>",
      "verbatim_text": "<complete, unsummarized text with [cite: N] markers>"
    }
  ],
  "continuation": {
    "continues_from_previous": "<true if content clearly continues from a prior page (e.g. mid-sentence, table header on previous page)>",
    "continues_to_next": "<true if content is clearly cut off and continues on the next page>",
    "continuation_type": "<null | text | table | list | narrative>"
  },
  "other_metadata": {}
}

PROCESSING RULES:
- STAKEHOLDER EXTRACTION (CRITICAL): Every grant document has AT LEAST two stakeholders: a funding agency/sponsor AND a recipient institution. ALWAYS extract both. The funding/granting agency (e.g. NSF, NIH, NHPRC, DOE) must be extracted as a stakeholder with role "Sponsor Contact" even if only mentioned in a header, award number prefix, or letterhead. The recipient institution must be extracted as role "Authorized Organizational Representative" or "Unknown". Also extract all individuals mentioned as investigators, collaborators, points of contact, or key personnel.
- NARRATIVE EXTRACTION (CRITICAL FOR RAG): Extract ALL body text, paragraphs, memos, and application answers VERBATIM to ensure 100% document coverage. If text is part of a Q&A form, include the question in prompt_or_header. For unstructured letter/memo body, use "General Body Text". Do NOT summarize, truncate, or condense.
- CITATIONS: Add [cite: N] numbered tags after each distinct statement in narrative text, incrementing N from 1.
- TABLE CLASSIFICATION (CRITICAL — strict format requirements):
  * Literal_Grid: irregular tables without clear headers.
    table_data = 2D array of strings, one inner array per row.
    Example: [["A", "B", "C"], ["D", "E", "F"]]

  * Key_Value_Form: label-value pairs (e.g. form cover sheets).
    table_data = single flat JSON object mapping each label to its value.
    Example: {"Name": "Alice", "ID": "123", "Date": "2024-01-01"}

  * Standard_Table: tabular data with clear column headers.
    table_data = array of objects. Each object = ONE ROW. The KEYS MUST BE
    the column header names (NOT the first cell's value). Every row object
    has the same set of keys matching the column headers.
    Example for a table with columns ["Report Type", "Frequency", "Copies", "URL"]:
      [
        {"Report Type": "Progress Report", "Frequency": "QR", "Copies": "1", "URL": "https://example.org"},
        {"Report Type": "Financial Report", "Frequency": "QR", "Copies": "1", "URL": "https://example.org"}
      ]
    DO NOT use the first column's value as the key. DO NOT emit bare strings
    between key-value pairs. Every value must belong to exactly one column-header key.
    If the table has merged/empty cells in the header row, use "" as the value
    for that column in that row.
- SIGNATURES: Do NOT read handwriting. Only note if a signature LINE exists and if a signature is DETECTED.
- STAKEHOLDER ROLES: Use ONLY the allowed stakeholder_role values listed above. If context does not make the role explicitly clear, use "Unknown". Capture raw_stakeholder_text verbatim.
- HYPERLINKS: If hyperlinks are provided in the context above, include them in the relevant narrative text or other_metadata. Preserve the exact URL.
- ADDRESSES: Use ONLY the allowed stakeholder_type values listed above. If unclear, use "Unknown". Place "Care Of"/"Attention" lines ONLY in care_of. Capture raw_address_text verbatim.
- Preserve ALL dollar amounts, dates, reference numbers exactly as they appear.
- CONTINUATION: If text, a table, or a list appears to be cut off at the top or bottom of the page, set the appropriate continuation flags. This helps downstream assembly merge cross-page content. If you cannot tell, set both to false.
- Missing fields: use null or "" as appropriate. Escape all strings.
- Output ONLY valid JSON. No markdown fences, no introductory text."""

In [ ]:
# ── Extraction prompt (per-page) ──────────────────────────────────────
# Adapted from the full document-level prompt for page-by-page processing.
# The assembly function merges per-page results into the final JSONL.

# ── Filename metadata parser ─────────────────────────────────────────
def parse_filename(filename):
    """Parse structured filename into FileNameMetaData.

    Rules from spec:
    - Six data elements in strict order, OFTEN split by single underscore.
    - Two underscores (__) next to each other = that data element is "".
    - After AwardID: Field2, Field3, Field4, Field5, then DocumentType.
    """
    import re
    stem = Path(filename).stem
    ext = Path(filename).suffix.lstrip('.')

    awd_match = re.search(r'_AWD-', stem)
    if not awd_match:
        return {
            "Drawer": stem, "AwardID": "", "Field2": "", "Field3": "",
            "Field4": "", "Field5": "", "DocumentType": stem,
            "DocumentTypeShort": stem, "FileType": ext
        }

    drawer = stem[:awd_match.start()]
    rest = stem[awd_match.start() + 1:]  # drop leading _

    parts = rest.split('_')
    award_id = parts[0] if parts else ""

    remaining = parts[1:]
    field2 = remaining[0] if len(remaining) > 0 else ""
    field3 = remaining[1] if len(remaining) > 1 else ""
    field4 = remaining[2] if len(remaining) > 2 else ""
    field5 = remaining[3] if len(remaining) > 3 else ""
    doc_type = '_'.join(remaining[4:]) if len(remaining) > 4 else ""

    doc_type_short = doc_type
    for cat in ("_Award_", "_Budget_", "_Report_", "_Agreement_", "_Proposal_"):
        idx = doc_type.find(cat)
        if idx >= 0:
            doc_type_short = doc_type[idx + len(cat):]
            break

    return {
        "Drawer": drawer, "AwardID": award_id,
        "Field2": field2, "Field3": field3, "Field4": field4, "Field5": field5,
        "DocumentType": doc_type, "DocumentTypeShort": doc_type_short,
        "FileType": ext
    }


# ── Document-level assembly ─────────────────────────────────────────────
# Takes per-page VLM outputs (in memory only) and produces the flat,
# Gemini-style monolithic document JSON with top-level aggregations:
# TablesCollection, Stakeholders, NarrativeResponses, AddressesCollection,
# etc. This is the only on-disk artifact for pass 1. Pass 2 adds a single
# DocumentSynthesis block on top of this same structure.
def assemble_document(filename, page_results, model_name, prompt_text=None, run_info=None):
    from datetime import datetime

    file_meta = parse_filename(filename)

    all_tables, all_narratives = [], []
    all_stakeholders, all_addresses = [], []
    all_tags = set()
    summaries = []
    has_annotation = has_watermark = False
    sig_info = {"PageNumber": None, "HasSignatureLine": False, "HasValidSignature": False}
    doc_details = {}
    other_meta = {}
    confidence_scores = []
    page_confidences = []

    for pr in page_results:
        pg = pr["page"]
        d = pr.get("extracted", {})

        for t in d.get("tables", []):
            all_tables.append({
                "PageNumber": pg,
                "TableClassification": t.get("table_classification",
                                             t.get("classification", "Standard_Table")),
                "TableData": t.get("table_data", t.get("rows", []))
            })

        for n in d.get("narrative_responses", []):
            all_narratives.append({
                "SectionOrPage": f"PAGE {pg}",
                "PromptOrHeader": n.get("prompt_or_header", ""),
                "VerbatimText": n.get("verbatim_text", "")
            })

        for s in d.get("stakeholders", []):
            if not any(v for v in s.values() if v):
                continue
            all_stakeholders.append({
                "PageNumber": pg,
                "ContextSnippet": s.get("context_snippet", ""),
                "StakeholderRole": s.get("stakeholder_role", "Unknown"),
                "FullName": s.get("full_name", ""),
                "FirstName": s.get("first_name", ""),
                "LastName": s.get("last_name", ""),
                "Email": s.get("email", ""),
                "Phone": s.get("phone", ""),
                "Institution": s.get("institution", ""),
                "Department": s.get("department", ""),
                "PositionTitle": s.get("position_title", ""),
                "HighestEducation": s.get("highest_education", ""),
                "RawStakeholderText": s.get("raw_stakeholder_text", "")
            })

        for a in d.get("addresses", []):
            if not any(v for v in a.values() if v):
                continue
            all_addresses.append({
                "PageNumber": pg,
                "ContextSnippet": a.get("context_snippet", ""),
                "Addressee": a.get("addressee", ""),
                "CareOf": a.get("care_of"),
                "AddressLine1": a.get("address_line1", ""),
                "AddressLine2": a.get("address_line2", ""),
                "City": a.get("city", ""),
                "StateProvince": a.get("state_province", ""),
                "PostalCode": a.get("postal_code", ""),
                "Country": a.get("country", ""),
                "StakeholderType": a.get("stakeholder_type", "Unknown"),
                "RawAddressText": a.get("raw_address_text", "")
            })

        all_tags.update(d.get("document_tags", []))
        summaries.append({
            "PageNumber": pg,
            "Summary": d.get("one_sentence_summary") or "N/A"
        })
        if d.get("has_annotation"): has_annotation = True
        if d.get("has_watermark"): has_watermark = True

        sig = d.get("signature_lines", {})
        if sig.get("has_signature_line"):
            sig_info = {"PageNumber": pg, "HasSignatureLine": True,
                        "HasValidSignature": sig.get("has_valid_signature", False)}

        if not doc_details and d.get("document_details"):
            doc_details = d["document_details"]

        if d.get("other_metadata"):
            other_meta.update(d["other_metadata"])

        conf_val = d.get("confidence_percentage")
        if isinstance(conf_val, (int, float)):
            confidence_scores.append(conf_val)
        page_confidences.append({
            "PageNumber": pg,
            "ConfidencePercentage": conf_val if isinstance(conf_val, (int, float)) else 0,
        })

    avg_conf = round(sum(confidence_scores) / len(confidence_scores), 1) if confidence_scores else 0.0
    page_times_ms = [pr.get("elapsed_ms", 0) for pr in page_results if pr.get("elapsed_ms")]
    timing = {
        "TotalElapsedSec": round(sum(page_times_ms) / 1000, 1) if page_times_ms else 0,
        "AvgPageSec": round(sum(page_times_ms) / len(page_times_ms) / 1000, 2) if page_times_ms else 0,
        "MinPageSec": round(min(page_times_ms) / 1000, 2) if page_times_ms else 0,
        "MaxPageSec": round(max(page_times_ms) / 1000, 2) if page_times_ms else 0,
    }

    now_iso = datetime.now().astimezone().isoformat(timespec="seconds")
    run_info = {
        "Model": model_name,
        "Timestamp": now_iso,
        "Timing": timing,
        **(run_info or {}),
        "ExtractionPrompt": prompt_text,
    }

    # Build extraction coverage manifest for QA
    method_counts = {}
    for pr in page_results:
        m = pr.get("method", "unknown")
        method_counts[m] = method_counts.get(m, 0) + 1

    _narr_pages = set()
    for n in all_narratives:
        sp = n.get("SectionOrPage", "")
        if sp.startswith("PAGE "):
            try:
                _narr_pages.add(int(sp.split()[-1]))
            except ValueError:
                pass

    thin_pages = [
        pr["page"] for pr in page_results
        if not pr.get("extracted", {}).get("tables")
        and not pr.get("extracted", {}).get("narrative_responses")
        and not pr.get("extracted", {}).get("stakeholders")
    ]

    extraction_coverage = {
        "TotalPages": len(page_results),
        "PagesWithTables": len(set(t["PageNumber"] for t in all_tables)),
        "PagesWithNarratives": len(_narr_pages),
        "PagesWithStakeholders": len(set(s["PageNumber"] for s in all_stakeholders)),
        "PagesWithParseErrors": sum(
            1 for pr in page_results
            if "parse_error" in pr.get("extracted", {})
        ),
        "ThinPages": thin_pages,
        "ThinPageCount": len(thin_pages),
        "MethodBreakdown": method_counts,
    }

    return {
        "Filename": filename,
        "FileNameMetaData": file_meta,
        "PageCount": len(page_results),
        "ConfidencePercentage": avg_conf,
        "PageConfidences": page_confidences,
        "LLMModelAndVersion": model_name,
        "CurrentDateAndTime": now_iso,
        "HasAnnotation": has_annotation,
        "HasWatermark": has_watermark,
        "SignatureLines": sig_info,
        "DocumentTags": sorted(all_tags),
        "OneSentenceNarrativeSummary": summaries,
        "DocumentDetails": {
            "ApplicationID": doc_details.get("application_id", ""),
            "ApplicationType": doc_details.get("application_type", ""),
            "Title": doc_details.get("title", ""),
            "RequestedAmount": doc_details.get("requested_amount"),
            "CompletedDate": doc_details.get("completed_date", ""),
            "SubDocumentType": doc_details.get("sub_document_type", "")
        },
        "Stakeholders": all_stakeholders,
        "AddressesCollection": all_addresses,
        "TablesCollection": all_tables,
        "NarrativeResponses": all_narratives,
        "OtherMetadata": other_meta,
        "ExtractionCoverage": extraction_coverage,
        "RunInfo": run_info,
    }


# Back-compat alias (old code referenced assemble_pass1_raw)
assemble_pass1_raw = assemble_document


> **Extracting library / archival materials instead?** Use the companion notebook
> `library_extraction_pipeline.ipynb` — same pipeline architecture (sliding window,
> two-pass, remote/local VLM mode) but with a library-specific prompt and schema
> (bibliographic metadata, body_text, marginalia, stamps/bookplates, musical
> notation, physical condition, page types). Don't try to mix schemas in this
> notebook — the assembly functions are grant-admin specific.


## 3. Load a sample document

Upload your sample PDFs/TIFFs to `/ocr/` using Jupyter's file upload button.

In [ ]:
ocr_dir = Path("/ocr")
files = [f for f in ocr_dir.iterdir() if f.is_file()]
print("Files in /ocr/:")
for f in sorted(files):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")
if not files:
    print("\nNo files found. Upload your sample docs to /ocr/ first.")

In [ ]:
DOC_PATH = Path("/ocr/" + f.name)

assert DOC_PATH.exists(), f"File not found: {DOC_PATH}"
print(f"Document: {DOC_PATH.name} ({DOC_PATH.stat().st_size / 1024:.0f} KB)")

## 4. Render pages

All pages are sent as images to the VLM (captures layout, tables, signatures, watermarks).

In [ ]:
import fitz
from PIL import Image

doc = fitz.open(str(DOC_PATH))
print(f"Pages: {len(doc)}\n")

page_images = []
page_links = []
for i, page in enumerate(doc):
    mat = fitz.Matrix(2.0, 2.0)  # 72 → 144 DPI: balance legibility vs. input-token cost
    pix = page.get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    page_images.append(img)

    links = extract_links(page)
    page_links.append(links)

    text = page.get_text("text").strip()
    has_text = len(text) >= 50
    label = "digital" if has_text else "scanned"
    link_info = f", {len(links)} link(s)" if links else ""
    print(f"Page {i+1}: {img.width}x{img.height} ({label}, {len(text)} chars{link_info})")

doc.close()
print(f"\n{len(page_images)} page(s) rendered. All will be sent as images to VLM.")


### Test extraction on a single page

Quick sanity check — renders one page and sends it to the VLM. Change `PAGE_IDX` to test different pages. Review the raw JSON output in the next cell before running the full batch.


In [ ]:
PAGE_IDX = 0  # Change this to test different pages
img = page_images[PAGE_IDX]

# Sliding window context (adjacent pages)
prev_img = page_images[PAGE_IDX - 1] if PAGE_IDX > 0 else None
next_img = page_images[PAGE_IDX + 1] if PAGE_IDX < len(page_images) - 1 else None
ctx_label = f" (with context: pages {PAGE_IDX if prev_img else ''}{'–' if prev_img and next_img else ''}{PAGE_IDX+2 if next_img else ''})" if (prev_img or next_img) else ""

print(f"Page {PAGE_IDX+1}: {img.width}x{img.height}{ctx_label}")
display(img.resize((img.width // 3, img.height // 3)))

t0 = time.time()
raw_result = extract_page(img, EXTRACTION_PROMPT, filename=DOC_PATH.name,
                          links=page_links[PAGE_IDX],
                          prev_image=prev_img, next_image=next_img)
elapsed = time.time() - t0
print(f"Extraction took {elapsed:.1f}s")


### Inspect the VLM's JSON output

Parses the raw VLM response (strips markdown fences if present) and pretty-prints the structured JSON. Check that tables, narratives, stakeholders etc. are being extracted correctly before running the batch.


In [ ]:
import json

# Strip markdown code fences if present
cleaned = raw_result.strip()
if cleaned.startswith("```"):
    cleaned = cleaned.split("\n", 1)[1]
    cleaned = cleaned.rsplit("```", 1)[0]

try:
    parsed = json.loads(cleaned)
    print("Valid JSON from VLM\n")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Invalid JSON: {e}\n")
    print("Raw output:")
    print(raw_result)

### Test filename metadata parser

Verifies that the structured filename is parsed correctly into Drawer, AwardID, Field2-5, DocumentType, and DocumentTypeShort. Compare against expected values.


In [ ]:
# Quick test: parse the current document filename
meta = parse_filename(DOC_PATH.name)
print("FileNameMetaData:")
for k, v in meta.items():
    print(f"  {k}: {v!r}")

## 8. Batch process all PDFs in /ocr/

Processes every PDF in the `/ocr/` folder and saves JSONL outputs.

### Configure paths

Set the directories for input PDFs, VLM output, and Gemini reference JSONs. The batch processor will find all `.pdf` files in `PDF_DIR` and save `.jsonl` outputs to `VLM_OUT_DIR`. The comparison step matches files by PDF filename stem.


In [ ]:
import json, fitz
from PIL import Image


def load_json_or_jsonl(path):
    """Load a JSON or JSONL file. Returns a dict (first object if JSONL/array)."""
    text = path.read_text().strip()
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data[0] if data else {}
        return data
    except json.JSONDecodeError:
        pass
    first_line = text.split("\n")[0].strip()
    data = json.loads(first_line)
    if isinstance(data, list):
        return data[0] if data else {}
    return data

# ── Configurable paths ────────────────────────────────────────────────
PDF_DIR = Path("/ocr")                          # PDFs to process
VLM_OUT_DIR = Path("/ocr/vlm_output")           # VLM JSONL output
GEMINI_DIR = Path("/ocr/gemini")                # Gemini reference JSONs

VLM_OUT_DIR.mkdir(exist_ok=True)

# Process smallest files first so quick wins land early in the batch.
pdf_files = sorted(PDF_DIR.glob("*.pdf"), key=lambda p: p.stat().st_size)
print(f"Found {len(pdf_files)} PDF(s) in {PDF_DIR}")
for p in pdf_files:
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

if GEMINI_DIR.exists():
    gemini_files = sorted(GEMINI_DIR.glob("*.json")) + sorted(GEMINI_DIR.glob("*.jsonl"))
    print(f"\nFound {len(gemini_files)} Gemini JSON(s) in {GEMINI_DIR}")
    for g in gemini_files:
        print(f"  {g.name}")
else:
    print(f"\nGemini dir not found: {GEMINI_DIR}")
    print("Create it and upload Gemini JSONs to enable comparison.")


### Pass 2: Document-level synthesis helper

Defines `run_pass2_synthesis()` and `apply_synthesis()`. These are called
**inline per-doc during the batch loop below**.

Pass 2 is strictly additive — it only inserts a `DocumentSynthesis` block
(title, type, creator, date, summary, cross-page notes) into the pass 1
monolithic JSON. It does not reshape, trim, or merge the pass 1 data.

**Output files per doc:**
- `{filename}_extracted.json` — flat Gemini-style document JSON.
  Pass 2 updates this file in place, adding `DocumentSynthesis` near
  the top. No separate `_final.json`.

**Set `RUN_PASS2 = False`** in the batch cell to skip pass 2.
**Set `RERUN_PASS2 = True`** in this cell to re-run synthesis over an
existing `batch_results` list after tweaking `SYNTHESIS_PROMPT`.

**Token budget:** `max_context_tokens=31000` by default (sized for a
vLLM server at `--max-model-len 32768`). Lower this at the call site if
your server is smaller. If the compacted pass 1 JSON exceeds the budget,
pass 2 is skipped and the pass 1 file stands as-is.


### Pass 2 document-level synthesis prompt (editable in its own cell)

Edit `SYNTHESIS_PROMPT` below to tweak what the VLM is asked to extract per page. Re-run this cell after changes — the batch loop uses `SYNTHESIS_PROMPT` directly.


In [ ]:
SYNTHESIS_PROMPT = """You are given the per-page extraction results from a multi-page document.
Each page was processed independently by a VLM. Your job is to produce
ONLY a document-level metadata summary. Do NOT rewrite, merge, or modify
the per-page content — it is the source of truth.

Return a JSON object with:
{
  "document_title": "<best title found across all pages>",
  "document_type": "<e.g. Notice of Award, Grant Guidelines, Book, Manuscript, Sheet Music Collection>",
  "creator": "<author, PI, composer, or agency — whoever created this document>",
  "date": "<most relevant date (publication, award, etc.)>",
  "language": "<primary language>",
  "page_count": <number>,
  "document_summary": "<2-3 sentence summary of the entire document>",
  "key_identifiers": {
    "award_number": "<if grant document>",
    "call_number": "<if library material>",
    "isbn_issn": "<if published work>"
  },
  "cross_page_notes": ["<any observations about content spanning pages, e.g. 'Budget table spans pages 3-5', 'Chapter 2 begins mid-page on p.12'>"]
}

RULES:
- Extract metadata from whichever page contains it (usually page 1 or title page)
- For document_summary, synthesize across ALL pages — do not just repeat page 1
- For cross_page_notes, look at the continuation flags in each page's JSON
- Output ONLY valid JSON. No markdown fences."""

In [ ]:
from pathlib import Path
import json

# ── Pass 2: Document-level synthesis ─────────────────────────────────
# Pass 2 adds a SINGLE DocumentSynthesis block on top of the pass 1
# monolithic JSON. No trimming cascade, no cross-page table stitching,
# no schema reshuffling. Pass 1 output stays untouched.
#
# Inputs to the VLM: the pass 1 aggregated JSON (minus raw text fields
# that waste tokens). If the compacted text still exceeds budget, we
# skip synthesis and keep the pass 1 file as the authoritative output.


def _compact_pass1_for_synthesis(pass1_output):
    """Drop verbose fields that add tokens without helping doc-level synthesis.

    Keeps every top-level field but strips per-item raw blobs (RawAddressText,
    RawStakeholderText, long VerbatimText) and the PageConfidences list, so
    the prompt to the synthesis VLM stays under the server budget.
    """
    compact = dict(pass1_output)
    compact.pop("PageConfidences", None)
    compact.pop("RunInfo", None)

    if compact.get("Stakeholders"):
        compact["Stakeholders"] = [
            {k: v for k, v in s.items()
             if k not in ("RawStakeholderText", "ContextSnippet")}
            for s in compact["Stakeholders"]
        ]
    if compact.get("AddressesCollection"):
        compact["AddressesCollection"] = [
            {k: v for k, v in a.items()
             if k not in ("RawAddressText", "ContextSnippet")}
            for a in compact["AddressesCollection"]
        ]
    if compact.get("NarrativeResponses"):
        compact["NarrativeResponses"] = [
            {
                "SectionOrPage": n.get("SectionOrPage", ""),
                "PromptOrHeader": n.get("PromptOrHeader", ""),
                "VerbatimText": (n.get("VerbatimText", "") or "")[:600],
            }
            for n in compact["NarrativeResponses"]
        ]
    if compact.get("TablesCollection"):
        # Keep classification + column headers; drop row bodies.
        slim_tables = []
        for t in compact["TablesCollection"]:
            data = t.get("TableData")
            entry = {
                "PageNumber": t.get("PageNumber"),
                "TableClassification": t.get("TableClassification"),
            }
            if isinstance(data, list) and data and isinstance(data[0], dict):
                entry["ColumnHeaders"] = list(data[0].keys())
                entry["RowCount"] = len(data)
            elif isinstance(data, list):
                entry["RowCount"] = len(data)
            elif isinstance(data, dict):
                entry["FormFieldCount"] = len(data)
            slim_tables.append(entry)
        compact["TablesCollection"] = slim_tables
    return compact


def run_pass2_synthesis(pass1_output, model_name, max_context_tokens=31000):
    """Ask the VLM for doc-level synthesis (title, type, summary, etc.).

    Budget: sized for a vLLM server at --max-model-len 32768 minus ~1.5K
    reserved for output. If the compacted pass 1 JSON still exceeds the
    budget, synthesis is skipped (returns None) and the pass 1 file is the
    final output.
    """
    compact = _compact_pass1_for_synthesis(pass1_output)
    body = json.dumps(compact, indent=1, ensure_ascii=False)
    est_tokens = len(body) // 4

    if est_tokens > max_context_tokens:
        print(f"  [pass2] SKIP: compacted pass 1 is ~{est_tokens} tokens "
              f"(> {max_context_tokens} budget).")
        return None

    messages = [
        {"role": "user", "content": [
            {"type": "text", "text": f"{SYNTHESIS_PROMPT}\n\n{body}"}
        ]}
    ]

    try:
        raw = run_vlm(messages, max_tokens=1500)
        cleaned = raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return json.loads(cleaned)
    except Exception as e:
        print(f"  [pass2] Synthesis failed: {e}")
        return None


def apply_synthesis(pass1_output, synthesis, pass2_elapsed):
    """Return a copy of the pass 1 dict with DocumentSynthesis inserted
    near the top and pass 2 bookkeeping appended to RunInfo. Caller is
    responsible for writing the result to disk once.
    """
    from datetime import datetime

    merged = {}
    for k, v in pass1_output.items():
        merged[k] = v
        if k == "PageCount" and synthesis is not None:
            merged["DocumentSynthesis"] = synthesis

    run_info = dict(merged.get("RunInfo") or {})
    run_info["Pass2ElapsedSec"] = round(pass2_elapsed, 1)
    run_info["Pass2Timestamp"] = datetime.now().astimezone().isoformat(timespec="seconds")
    run_info["SynthesisPrompt"] = SYNTHESIS_PROMPT
    merged["RunInfo"] = run_info

    return merged


print("Pass 2 functions ready: run_pass2_synthesis(), apply_synthesis()")

# Optional: re-run pass 2 over batch_results if needed (e.g. after tweaking prompt)
RERUN_PASS2 = False
if RERUN_PASS2 and batch_results:
    print(f"\nRe-running pass 2 on {len(batch_results)} documents...\n")
    for br in batch_results:
        if "output" not in br:
            continue
        t0 = time.time()
        synthesis = run_pass2_synthesis(br["output"], MODEL_NAME)
        elapsed = time.time() - t0
        if synthesis:
            br["synthesis"] = synthesis
            merged = apply_synthesis(br["output"], synthesis, elapsed)
            br["output"] = merged
            if br.get("output_path"):
                Path(br["output_path"]).write_text(
                    json.dumps(merged, indent=2, ensure_ascii=False) + "\n")
                print(f"  {br['filename']}: OK ({elapsed:.1f}s) -> {Path(br['output_path']).name}")
        else:
            print(f"  {br['filename']}: skipped ({elapsed:.1f}s)")


### Run batch extraction

Loops through every PDF in the input directory. For each document:
1. Renders all pages at 1x resolution
2. Sends each page image to the VLM with the extraction prompt
3. Parses the JSON response (handles markdown fences and parse failures)
4. Assembles page results into a document-level JSONL and saves to disk

Progress is printed per-page with timing.


In [ ]:
import json, fitz, time
from pathlib import Path
from PIL import Image

SKIP_EXISTING = True          # Skip any doc whose _extracted.json already exists
USE_SLIDING_WINDOW = True     # 3-page context per call
RUN_PASS2 = True              # Run document-level synthesis
CLEAN_PAGE_CACHE_ON_DONE = True  # Delete <stem>_pages/ once the final JSON is written

batch_results = []
skipped = 0

for pdf_path in pdf_files:
    out_path = VLM_OUT_DIR / f"{pdf_path.stem}_extracted.json"
    legacy_jsonl = VLM_OUT_DIR / f"{pdf_path.stem}_extracted.jsonl"
    if legacy_jsonl.exists() and not out_path.exists():
        out_path = legacy_jsonl

    # Per-page cache dir: one JSON per successfully extracted page. Lets a
    # failed run resume page-by-page instead of re-doing the whole doc.
    pages_dir = VLM_OUT_DIR / f"{pdf_path.stem}_pages"

    if SKIP_EXISTING and out_path.exists():
        skipped += 1
        batch_results.append({"filename": pdf_path.name,
                              "output": load_json_or_jsonl(out_path),
                              "output_path": str(out_path)})
        print(f"SKIP (done): {pdf_path.name}")
        continue

    print(f"\n{'='*60}")
    print(f"Processing: {pdf_path.name}")
    print(f"{'='*60}")

    # Report any partial progress from an earlier failed run.
    if pages_dir.exists():
        cached = sorted(pages_dir.glob("page_*.json"))
        if cached:
            print(f"  Resuming: {len(cached)} page(s) already cached in {pages_dir.name}/")

    try:
        doc = fitz.open(str(pdf_path))
        page_count = len(doc)

        images = []
        links_per_page = []
        for i, page in enumerate(doc):
            mat = fitz.Matrix(2.0, 2.0)  # 72 → 144 DPI: balance legibility vs. input-token cost
            pix = page.get_pixmap(matrix=mat)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            images.append(img)
            links_per_page.append(extract_links(page))
        doc.close()
        mode = "sliding window (3-page)" if USE_SLIDING_WINDOW else "single page"
        print(f"  {page_count} page(s) rendered — extraction mode: {mode}")

        pages_dir.mkdir(exist_ok=True)
        page_results = []
        missing_pages = []  # pages that still failed on this run

        for i, img in enumerate(images):
            page_num = i + 1
            page_file = pages_dir / f"page_{page_num:03d}.json"

            # ── Resume: reuse cached page if present ─────────────────────
            if page_file.exists():
                try:
                    cached_record = json.loads(page_file.read_text())
                    page_results.append(cached_record)
                    cached_method = cached_record.get("method", "vlm_image")
                    print(f"    Page {page_num}/{page_count}: loaded from cache ({cached_method})")
                    continue
                except json.JSONDecodeError:
                    print(f"    Page {page_num}: cached file corrupt — re-extracting")
                    page_file.unlink()

            # ── Extract: 3 fallback tiers ─────────────────────────────────
            # Tier 1: sliding window, 2x DPI (full quality)
            # Tier 2: single page,    2x DPI (no context images — fewer tokens)
            # Tier 3: single page,    1x DPI (gateway-timeout buster — ~75% fewer
            #         image tokens so generation fits under the ingress timeout)
            t0 = time.time()
            raw = None
            method = "vlm_image"
            page_error = None

            def _render_page_1x(pdf_path, page_idx):
                """Re-render a single page at native DPI (72dpi) for the low-res fallback."""
                import fitz as _fitz
                _doc = _fitz.open(str(pdf_path))
                _page = _doc[page_idx]
                _pix = _page.get_pixmap()   # default matrix = 1x / 72dpi
                _img = Image.frombytes("RGB", [_pix.width, _pix.height], _pix.samples)
                _doc.close()
                return _img

            # Cap max_tokens hard at tier-3. A 504 after the full 300s httpx
            # timeout means the VLM is actively generating but not finishing
            # — usually a token loop on decorative content. Lower max_tokens
            # forces it to stop emitting and return, which also ends the
            # gateway's wait. 4096 is enough for a typical dense page.
            _T3_MAX_TOKENS = 4096

            # Orientation-aware logging: flag transitions for QA
            is_landscape = img.size[0] > img.size[1]
            prev_landscape = images[i - 1].size[0] > images[i - 1].size[1] if i > 0 else is_landscape
            if is_landscape != prev_landscape:
                orient_from = "portrait" if prev_landscape else "landscape"
                orient_to = "landscape" if is_landscape else "portrait"
                print(f"    Page {page_num}: orientation change ({orient_from} → {orient_to})")

            try:
                if USE_SLIDING_WINDOW:
                    prev_img = images[i - 1] if i > 0 else None
                    next_img = images[i + 1] if i < len(images) - 1 else None
                    raw = extract_page(img, EXTRACTION_PROMPT, filename=pdf_path.name,
                                       links=links_per_page[i],
                                       prev_image=prev_img, next_image=next_img)
                else:
                    raw = extract_page(img, EXTRACTION_PROMPT, filename=pdf_path.name,
                                       links=links_per_page[i])
            except Exception as e:
                print(f"    Page {page_num}: tier-1 failed ({type(e).__name__}: {e})")
                # Tier 2: single page, same 2x image
                print(f"    Page {page_num}: tier-2 — single page, 2x DPI")
                try:
                    raw = extract_page(img, EXTRACTION_PROMPT, filename=pdf_path.name,
                                       links=links_per_page[i])
                    method = "vlm_image_fallback_t2"
                except Exception as e2:
                    print(f"    Page {page_num}: tier-2 failed ({type(e2).__name__}: {e2})")
                    # Tier 3: re-render at 1x DPI + cap max_tokens at 4096.
                    # ~75% fewer image tokens AND hard output cap means
                    # generation must finish well before the gateway timeout.
                    print(f"    Page {page_num}: tier-3 — single page, 1x DPI, max_tokens={_T3_MAX_TOKENS}")
                    try:
                        img_1x = _render_page_1x(pdf_path, i)
                        raw = extract_page(img_1x, EXTRACTION_PROMPT, filename=pdf_path.name,
                                           links=links_per_page[i],
                                           max_tokens=_T3_MAX_TOKENS)
                        method = "vlm_image_fallback_t3_lowres"
                    except Exception as e3:
                        page_error = f"{type(e3).__name__}: {e3}"
                        print(f"    Page {page_num}: tier-3 also failed ({page_error})")
                        # Save the page image so we can actually look at what's
                        # tripping the model. Sometimes a page is just a
                        # full-page logo/background that causes a generation
                        # loop — easier to diagnose with the PNG in hand.
                        try:
                            fail_dir = VLM_OUT_DIR / "extraction_failures"
                            fail_dir.mkdir(exist_ok=True)
                            png_path = fail_dir / f"{pdf_path.stem}_page{page_num:03d}.png"
                            img.save(png_path)
                            print(f"    Page {page_num}: saved {png_path.name} for inspection")
                        except Exception as save_err:
                            print(f"    Page {page_num}: (could not save debug image: {save_err})")

            elapsed = time.time() - t0

            if raw is None:
                # Hard extraction failure — DO NOT cache this page. Next run
                # will retry only these missing pages.
                missing_pages.append(page_num)
                print(f"    Page {page_num}/{page_count}: EXTRACTION FAILED ({elapsed:.1f}s) — will retry on next run")
                continue

            extracted, parse_err = parse_vlm_json(raw)
            if extracted is None:
                # Parse failure is deterministic — not worth retrying. Cache
                # the raw output so we keep what the model gave us.
                print(f"    Page {page_num}: PARSE FAILED ({parse_err})")
                debug_dir = VLM_OUT_DIR / "parse_failures"
                debug_dir.mkdir(exist_ok=True)
                debug_path = debug_dir / f"{pdf_path.stem}_page{page_num:03d}_raw.txt"
                debug_path.write_text(raw, encoding="utf-8")
                print(f"      Raw output saved: {debug_path.name}")
                extracted = {
                    "raw_text": raw,
                    "parse_error": parse_err,
                    "confidence_percentage": 0,
                    "confidence_narrative": f"Failed to parse structured output ({parse_err})",
                }

            record = {"page": page_num, "method": method,
                      "elapsed_ms": round(elapsed * 1000, 1), "extracted": extracted}
            page_file.write_text(json.dumps(record, indent=2, ensure_ascii=False))
            page_results.append(record)
            ctx = "+ctx" if (USE_SLIDING_WINDOW and method == "vlm_image") else ("↓t2" if method == "vlm_image_fallback_t2" else ("↓t3" if method == "vlm_image_fallback_t3_lowres" else ""))
            print(f"    Page {page_num}/{page_count}{ctx}: {elapsed:.1f}s  → {page_file.name}")

        # ── Don't write the final JSON if any page is still missing ─────
        if missing_pages:
            print(f"  INCOMPLETE: {len(missing_pages)} page(s) still missing: {missing_pages}")
            print(f"  Cached pages preserved in {pages_dir}/ — re-run this cell to retry only the missing pages.")
            print(f"  Final _extracted.json NOT written (doc will stay in the queue).")
            continue

        # Sort so page_results is in page order regardless of retry order
        page_results.sort(key=lambda r: r["page"])

        # Assemble pass 1 (flat Gemini-style output)
        run_info = {
            "VLMMode": VLM_MODE,
            "VLLMEndpoint": VLLM_BASE_URL if VLM_MODE == "remote" else None,
            "UseSlidingWindow": USE_SLIDING_WINDOW,
        }
        output = assemble_document(pdf_path.name, page_results, MODEL_NAME,
                                   prompt_text=EXTRACTION_PROMPT, run_info=run_info)

        br = {"filename": pdf_path.name, "output": output,
              "output_path": str(out_path)}

        # Pass 2 (optional) — merges in place before the first write to disk,
        # so "file exists" = "fully done"
        if RUN_PASS2:
            t2 = time.time()
            synthesis = run_pass2_synthesis(output, MODEL_NAME)
            elapsed2 = time.time() - t2
            if synthesis:
                br["synthesis"] = synthesis
                output = apply_synthesis(output, synthesis, elapsed2)
                br["output"] = output
                print(f"  Synthesis merged ({elapsed2:.1f}s) — {synthesis.get('document_type', '?')}")
            else:
                print(f"  Pass 2 skipped/failed ({elapsed2:.1f}s) — saving pass 1 only")

        # Single write at the end
        out_path.write_text(json.dumps(output, indent=2, ensure_ascii=False) + "\n")
        print(f"  Saved: {out_path.name}")

        # Clean up the per-page cache now that the final file is written
        if CLEAN_PAGE_CACHE_ON_DONE and pages_dir.exists():
            for p in pages_dir.glob("*"):
                p.unlink()
            try:
                pages_dir.rmdir()
            except OSError:
                pass  # not empty (e.g. stray files) — leave it

        batch_results.append(br)
    except Exception as e:
        # Infrastructure-level failure (OOM rendering, fitz crash, etc).
        # Per-page cache is preserved so the next run resumes where we left off.
        print(f"  FAILED ({type(e).__name__}): {e}")
        print(f"  Cached pages (if any) preserved in {pages_dir}/ — re-run to resume.")
        continue

print(f"\n{'='*60}")
print(f"Batch complete. {len(batch_results)} doc(s) total.")
print(f"  Skipped (already done): {skipped}")
print(f"  Fresh processed: {len(batch_results) - skipped}")
print(f"Output dir: {VLM_OUT_DIR}")


## 9. Zip the output folder for download

Bundle `vlm_output/` into a timestamped zip next to the folder so
you can right-click → Download it from the JupyterLab file browser.
Re-running drops a new timestamped archive instead of overwriting
the previous one.


In [ ]:
import shutil, time
from pathlib import Path

archive_base = VLM_OUT_DIR.parent / f"vlm_output_{time.strftime('%Y%m%d_%H%M%S')}"
archive_path = shutil.make_archive(
    base_name=str(archive_base),   # no extension — make_archive adds .zip
    format="zip",
    root_dir=VLM_OUT_DIR.parent,
    base_dir=VLM_OUT_DIR.name,     # paths inside zip are rooted at vlm_output/
)
size_mb = Path(archive_path).stat().st_size / 1024**2
print(f"Archive: {archive_path}  ({size_mb:.1f} MB)")


## 10. Compare VLM vs Gemini extractions

Side-by-side comparison of what our local VLM extracted vs what Gemini produced for the same PDFs. This tells us whether the local model is competitive with Gemini for grant document extraction.

**How it works:**
1. Matches files by PDF name (e.g. `foo.pdf` -> VLM's `foo_extracted.jsonl` vs Gemini's `foo.json`)
2. For each matched pair, compares every section of the output schema
3. Produces a per-document text report + 6-panel visual comparison

**What the metrics mean:**

| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **Narrative text coverage** | Total characters of verbatim text extracted | More = the model captured more of the actual document content. Critical for RAG — missing text means missing search results. |
| **Text similarity** | How much the VLM and Gemini narrative text overlap (0-100%) | High = both models extracted the same content. Low = one model found text the other missed, OR they structured it differently. |
| **Table cells** | Number of individual data cells extracted from tables | More cells = more complete table extraction. Missing cells = lost dollar amounts, dates, etc. |
| **Stakeholders** | People/orgs identified (PIs, contacts, sponsors) | Checks if the model finds all named individuals and their roles. |
| **Addresses** | Physical addresses found in the document | Checks if letterhead, mailing, and signature block addresses are captured. |
| **Confidence** | The model's self-reported extraction confidence | Self-assessed, so take with a grain of salt. Useful for flagging pages the model struggled with. |
| **Wins scorecard** | Which model "won" more categories across all documents | Quick summary: is VLM or Gemini better overall? |


### Comparison functions

Defines the comparison and plotting code. Run this cell to load the functions — the next cell actually executes the comparison.


In [ ]:
import json
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import numpy as np

def compare_extractions(vlm, gem, filename):
    """Compare VLM vs Gemini extraction for a single document. Returns report dict."""
    report = {"filename": filename, "sections": {}}

    # ── Narrative coverage
    vlm_narr = vlm.get("NarrativeResponses", [])
    gem_narr = gem.get("NarrativeResponses", [])
    vlm_text = " ".join(n.get("VerbatimText", "") for n in vlm_narr)
    gem_text = " ".join(n.get("VerbatimText", "") for n in gem_narr)
    text_sim = SequenceMatcher(None, vlm_text.lower(), gem_text.lower()).ratio() if (vlm_text and gem_text) else 0

    report["sections"]["NarrativeResponses"] = {
        "vlm_count": len(vlm_narr), "gemini_count": len(gem_narr),
        "vlm_total_chars": len(vlm_text), "gemini_total_chars": len(gem_text),
        "text_similarity": round(text_sim * 100, 1),
        "vlm_has_citations": "[cite:" in vlm_text,
        "gemini_has_citations": "[cite:" in gem_text,
    }

    # ── Tables
    vlm_tables = vlm.get("TablesCollection", [])
    gem_tables = gem.get("TablesCollection", [])

    def count_table_cells(tables):
        total = 0
        for t in tables:
            td = t.get("TableData", [])
            if isinstance(td, list):
                for row in td:
                    total += len(row) if isinstance(row, (dict, list)) else 1
            elif isinstance(td, dict):
                total += len(td)
        return total

    report["sections"]["TablesCollection"] = {
        "vlm_count": len(vlm_tables), "gemini_count": len(gem_tables),
        "vlm_total_cells": count_table_cells(vlm_tables),
        "gemini_total_cells": count_table_cells(gem_tables),
        "vlm_classifications": [t.get("TableClassification", "?") for t in vlm_tables],
        "gemini_classifications": [t.get("TableClassification", "?") for t in gem_tables],
    }

    # ── Stakeholders
    vlm_stak = vlm.get("Stakeholders", [])
    gem_stak = gem.get("Stakeholders", [])
    vlm_names = sorted(set(s.get("FullName", "") for s in vlm_stak if s.get("FullName")))
    gem_names = sorted(set(s.get("FullName", "") for s in gem_stak if s.get("FullName")))

    report["sections"]["Stakeholders"] = {
        "vlm_count": len(vlm_stak), "gemini_count": len(gem_stak),
        "vlm_names": vlm_names, "gemini_names": gem_names,
        "names_in_common": sorted(set(vlm_names) & set(gem_names)),
        "vlm_only": sorted(set(vlm_names) - set(gem_names)),
        "gemini_only": sorted(set(gem_names) - set(vlm_names)),
    }

    # ── Addresses
    vlm_addr = vlm.get("AddressesCollection", [])
    gem_addr = gem.get("AddressesCollection", [])
    report["sections"]["AddressesCollection"] = {
        "vlm_count": len(vlm_addr), "gemini_count": len(gem_addr),
    }

    # ── Document tags
    vlm_tags = set(vlm.get("DocumentTags", []))
    gem_tags = set(gem.get("DocumentTags", []))
    report["sections"]["DocumentTags"] = {
        "vlm_tags": sorted(vlm_tags), "gemini_tags": sorted(gem_tags),
        "in_common": sorted(vlm_tags & gem_tags),
        "vlm_only": sorted(vlm_tags - gem_tags),
        "gemini_only": sorted(gem_tags - vlm_tags),
    }

    # ── Document details
    vlm_dd = vlm.get("DocumentDetails", {})
    gem_dd = gem.get("DocumentDetails", {})
    dd_diffs = {}
    for key in set(list(vlm_dd.keys()) + list(gem_dd.keys())):
        v, g = vlm_dd.get(key, ""), gem_dd.get(key, "")
        if str(v).strip() != str(g).strip():
            dd_diffs[key] = {"vlm": v, "gemini": g}

    report["sections"]["DocumentDetails"] = {
        "matching_fields": len(vlm_dd) - len(dd_diffs),
        "differing_fields": dd_diffs,
    }

    # ── Flags
    report["sections"]["Flags"] = {
        "HasAnnotation": {"vlm": vlm.get("HasAnnotation"), "gemini": gem.get("HasAnnotation")},
        "HasWatermark": {"vlm": vlm.get("HasWatermark"), "gemini": gem.get("HasWatermark")},
        "SignatureLines": {"vlm": vlm.get("SignatureLines"), "gemini": gem.get("SignatureLines")},
    }

    # ── Confidence
    report["sections"]["Confidence"] = {
        "vlm": vlm.get("ConfidencePercentage"),
        "gemini": gem.get("ConfidencePercentage"),
        "vlm_narrative": vlm.get("ConfidenceNarrative", "")[:200],
        "gemini_narrative": gem.get("ConfidenceNarrative", "")[:200],
    }

    # ── FileNameMetaData
    vlm_fm = vlm.get("FileNameMetaData", {})
    gem_fm = gem.get("FileNameMetaData", {})
    fm_match = sum(1 for k in vlm_fm if str(vlm_fm.get(k, "")) == str(gem_fm.get(k, "")))

    report["sections"]["FileNameMetaData"] = {
        "matching": fm_match, "total": max(len(vlm_fm), len(gem_fm)),
        "diffs": {k: {"vlm": vlm_fm.get(k, ""), "gemini": gem_fm.get(k, "")}
                  for k in set(list(vlm_fm.keys()) + list(gem_fm.keys()))
                  if str(vlm_fm.get(k, "")) != str(gem_fm.get(k, ""))},
    }

    return report


def print_report(report):
    """Pretty-print a comparison report."""
    print(f"\n{'='*70}")
    print(f"  {report['filename']}")
    print(f"{'='*70}")
    s = report["sections"]

    n = s["NarrativeResponses"]
    print(f"\n  NARRATIVES")
    print(f"    Sections:   VLM={n['vlm_count']}, Gemini={n['gemini_count']}")
    print(f"    Text chars: VLM={n['vlm_total_chars']:,}, Gemini={n['gemini_total_chars']:,}")
    print(f"    Similarity: {n['text_similarity']}%")
    print(f"    Citations:  VLM={'yes' if n['vlm_has_citations'] else 'NO'}, "
          f"Gemini={'yes' if n['gemini_has_citations'] else 'NO'}")
    winner = "VLM" if n["vlm_total_chars"] >= n["gemini_total_chars"] else "Gemini"
    print(f"    >> More text coverage: {winner}")

    t = s["TablesCollection"]
    print(f"\n  TABLES")
    print(f"    Count: VLM={t['vlm_count']}, Gemini={t['gemini_count']}")
    print(f"    Cells: VLM={t['vlm_total_cells']}, Gemini={t['gemini_total_cells']}")

    sk = s["Stakeholders"]
    print(f"\n  STAKEHOLDERS")
    print(f"    Count: VLM={sk['vlm_count']}, Gemini={sk['gemini_count']}")
    if sk["names_in_common"]: print(f"    Common:      {sk['names_in_common']}")
    if sk["vlm_only"]: print(f"    VLM only:    {sk['vlm_only']}")
    if sk["gemini_only"]: print(f"    Gemini only: {sk['gemini_only']}")

    a = s["AddressesCollection"]
    print(f"\n  ADDRESSES: VLM={a['vlm_count']}, Gemini={a['gemini_count']}")

    dd = s["DocumentDetails"]
    print(f"\n  DOCUMENT DETAILS: {dd['matching_fields']} matching")
    for k, v in dd["differing_fields"].items():
        print(f"    {k}: VLM={v['vlm']!r} vs Gemini={v['gemini']!r}")

    c = s["Confidence"]
    print(f"\n  CONFIDENCE: VLM={c['vlm']}%, Gemini={c['gemini']}%")

    fm = s["FileNameMetaData"]
    print(f"  FILENAME METADATA: {fm['matching']}/{fm['total']} matching")


def plot_comparison(reports, model_name):
    """Generate comparison plots for VLM vs Gemini."""
    if not reports:
        print("No reports to plot.")
        return

    doc_names = [r["filename"][:40] for r in reports]
    n_docs = len(reports)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"VLM ({model_name.split('/')[-1]}) vs Gemini", fontsize=14, fontweight="bold")

    colors_vlm = "#2196F3"
    colors_gem = "#FF9800"

    # 1. Narrative text coverage (chars)
    ax = axes[0, 0]
    vlm_chars = [r["sections"]["NarrativeResponses"]["vlm_total_chars"] for r in reports]
    gem_chars = [r["sections"]["NarrativeResponses"]["gemini_total_chars"] for r in reports]
    x = np.arange(n_docs)
    w = 0.35
    ax.barh(x - w/2, vlm_chars, w, label="VLM", color=colors_vlm)
    ax.barh(x + w/2, gem_chars, w, label="Gemini", color=colors_gem)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Characters")
    ax.set_title("Narrative Text Coverage")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    # 2. Text similarity
    ax = axes[0, 1]
    sims = [r["sections"]["NarrativeResponses"]["text_similarity"] for r in reports]
    bars = ax.barh(x, sims, color=["#4CAF50" if s >= 70 else "#FFC107" if s >= 40 else "#F44336" for s in sims])
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Similarity %")
    ax.set_title("Narrative Text Similarity")
    ax.set_xlim(0, 100)
    ax.axvline(70, color="gray", linestyle="--", alpha=0.5)
    ax.invert_yaxis()

    # 3. Table cells
    ax = axes[0, 2]
    vlm_cells = [r["sections"]["TablesCollection"]["vlm_total_cells"] for r in reports]
    gem_cells = [r["sections"]["TablesCollection"]["gemini_total_cells"] for r in reports]
    ax.barh(x - w/2, vlm_cells, w, label="VLM", color=colors_vlm)
    ax.barh(x + w/2, gem_cells, w, label="Gemini", color=colors_gem)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Cells extracted")
    ax.set_title("Table Extraction (cells)")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    # 4. Stakeholders + Addresses
    ax = axes[1, 0]
    vlm_sk = [r["sections"]["Stakeholders"]["vlm_count"] for r in reports]
    gem_sk = [r["sections"]["Stakeholders"]["gemini_count"] for r in reports]
    vlm_ad = [r["sections"]["AddressesCollection"]["vlm_count"] for r in reports]
    gem_ad = [r["sections"]["AddressesCollection"]["gemini_count"] for r in reports]
    w2 = 0.2
    ax.barh(x - 1.5*w2, vlm_sk, w2, label="VLM Stakeholders", color=colors_vlm)
    ax.barh(x - 0.5*w2, gem_sk, w2, label="Gemini Stakeholders", color=colors_gem)
    ax.barh(x + 0.5*w2, vlm_ad, w2, label="VLM Addresses", color=colors_vlm, alpha=0.5)
    ax.barh(x + 1.5*w2, gem_ad, w2, label="Gemini Addresses", color=colors_gem, alpha=0.5)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Count")
    ax.set_title("Stakeholders & Addresses")
    ax.legend(fontsize=7)
    ax.invert_yaxis()

    # 5. Confidence scores
    ax = axes[1, 1]
    vlm_conf = [r["sections"]["Confidence"]["vlm"] or 0 for r in reports]
    gem_conf = [r["sections"]["Confidence"]["gemini"] or 0 for r in reports]
    ax.barh(x - w/2, vlm_conf, w, label="VLM", color=colors_vlm)
    ax.barh(x + w/2, gem_conf, w, label="Gemini", color=colors_gem)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Confidence %")
    ax.set_title("Self-Reported Confidence")
    ax.set_xlim(0, 100)
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    # 6. Wins scorecard
    ax = axes[1, 2]
    categories = ["Narratives", "Tables", "Stakeholders", "Addresses"]
    vlm_w = [0, 0, 0, 0]
    gem_w = [0, 0, 0, 0]
    for r in reports:
        s = r["sections"]
        n = s["NarrativeResponses"]
        if n["vlm_total_chars"] > n["gemini_total_chars"]: vlm_w[0] += 1
        elif n["gemini_total_chars"] > n["vlm_total_chars"]: gem_w[0] += 1
        t = s["TablesCollection"]
        if t["vlm_total_cells"] > t["gemini_total_cells"]: vlm_w[1] += 1
        elif t["gemini_total_cells"] > t["vlm_total_cells"]: gem_w[1] += 1
        sk = s["Stakeholders"]
        if sk["vlm_count"] > sk["gemini_count"]: vlm_w[2] += 1
        elif sk["gemini_count"] > sk["vlm_count"]: gem_w[2] += 1
        a = s["AddressesCollection"]
        if a["vlm_count"] > a["gemini_count"]: vlm_w[3] += 1
        elif a["gemini_count"] > a["vlm_count"]: gem_w[3] += 1
    y = np.arange(len(categories))
    ax.barh(y - w/2, vlm_w, w, label="VLM wins", color=colors_vlm)
    ax.barh(y + w/2, gem_w, w, label="Gemini wins", color=colors_gem)
    ax.set_yticks(y)
    ax.set_yticklabels(categories)
    ax.set_xlabel("Documents won")
    ax.set_title("Wins Scorecard")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    plt.tight_layout()
    plt.show()

print("Comparison functions ready (with plots).")


### Run comparison and generate plots

Runs the comparison for all matched VLM/Gemini pairs and prints:
- **Per-document report**: detailed breakdown of each section (narratives, tables, stakeholders, etc.)
- **Aggregate scorecard**: tallies which model wins each category across all documents
- **6-panel chart**: visual comparison — narrative coverage, text similarity, table completeness, stakeholders/addresses, confidence, and overall wins


In [ ]:
# ── Run comparison ────────────────────────────────────────────────────
reports = []
matched = 0
unmatched_vlm = []
unmatched_gem = []

# Build lookup: PDF stem -> VLM output
vlm_files = {p.stem.replace("_extracted", ""): p
             for p in VLM_OUT_DIR.glob("*.jsonl")}

# Build lookup: PDF stem -> Gemini output (try multiple naming conventions)
gem_lookup = {}
if GEMINI_DIR.exists():
    for g in list(GEMINI_DIR.glob("*.json")) + list(GEMINI_DIR.glob("*.jsonl")):
        stem = g.stem.replace("_extracted", "").replace("_gemini", "")
        gem_lookup[stem] = g

# Match and compare
for stem, vlm_path in sorted(vlm_files.items()):
    if stem in gem_lookup:
        matched += 1
        vlm_data = load_json_or_jsonl(vlm_path)
        gem_data = load_json_or_jsonl(gem_lookup[stem])
        report = compare_extractions(vlm_data, gem_data, vlm_data.get("Filename", stem))
        reports.append(report)
        print_report(report)
    else:
        unmatched_vlm.append(stem)

for stem in gem_lookup:
    if stem not in vlm_files:
        unmatched_gem.append(stem)

# ── Aggregate summary ─────────────────────────────────────────────────
print(f"\n\n{'#'*70}")
print(f"  AGGREGATE SUMMARY — {len(reports)} document(s) compared")
print(f"{'#'*70}")

if unmatched_vlm:
    print(f"\n  VLM outputs with no Gemini match: {unmatched_vlm}")
if unmatched_gem:
    print(f"  Gemini JSONs with no VLM match: {unmatched_gem}")

if reports:
    # Tally wins per section
    vlm_wins = {"narratives": 0, "tables": 0, "stakeholders": 0, "addresses": 0}
    gem_wins = {"narratives": 0, "tables": 0, "stakeholders": 0, "addresses": 0}
    total_sim = 0

    for r in reports:
        s = r["sections"]
        n = s["NarrativeResponses"]
        total_sim += n["text_similarity"]
        if n["vlm_total_chars"] > n["gemini_total_chars"]:
            vlm_wins["narratives"] += 1
        elif n["gemini_total_chars"] > n["vlm_total_chars"]:
            gem_wins["narratives"] += 1

        t = s["TablesCollection"]
        if t["vlm_total_cells"] > t["gemini_total_cells"]:
            vlm_wins["tables"] += 1
        elif t["gemini_total_cells"] > t["vlm_total_cells"]:
            gem_wins["tables"] += 1

        sk = s["Stakeholders"]
        if sk["vlm_count"] > sk["gemini_count"]:
            vlm_wins["stakeholders"] += 1
        elif sk["gemini_count"] > sk["vlm_count"]:
            gem_wins["stakeholders"] += 1

        a = s["AddressesCollection"]
        if a["vlm_count"] > a["gemini_count"]:
            vlm_wins["addresses"] += 1
        elif a["gemini_count"] > a["vlm_count"]:
            gem_wins["addresses"] += 1

    n_docs = len(reports)
    print(f"\n  Average narrative text similarity: {total_sim / n_docs:.1f}%")
    print(f"\n  {'Category':<20} {'VLM wins':<12} {'Gemini wins':<12} {'Tie':<8}")
    print(f"  {'-'*52}")
    for cat in vlm_wins:
        ties = n_docs - vlm_wins[cat] - gem_wins[cat]
        print(f"  {cat:<20} {vlm_wins[cat]:<12} {gem_wins[cat]:<12} {ties:<8}")

    total_vlm = sum(vlm_wins.values())
    total_gem = sum(gem_wins.values())
    print(f"  {'-'*52}")
    print(f"  {'TOTAL':<20} {total_vlm:<12} {total_gem:<12} "
          f"{n_docs * 4 - total_vlm - total_gem:<8}")

    if total_vlm > total_gem:
        print(f"\n  >> Overall winner: VLM ({MODEL_NAME})")
    elif total_gem > total_vlm:
        print(f"\n  >> Overall winner: Gemini")
    else:
        print(f"\n  >> Result: TIE")
else:
    print("\n  No matched documents to compare.")
    print("  Upload Gemini JSONs to /ocr/gemini/ with filenames matching the PDFs.")

# ── Plots ─────────────────────────────────────────────────────────────
plot_comparison(reports, MODEL_NAME)


## 11. Launch Streamlit app

Tests the extraction server + Streamlit UI stack from within this notebook — same code that runs in production as separate jobs.

**Prerequisites:**
1. Your workspace must have a **Custom URL tool on port 8501** configured (in addition to Jupyter on 8888). Ask your admin to add this if you don't have it.
2. Set env var `STREAMLIT_BASE_PATH` to the tool's URL path. Check the RunAI UI — click the Custom URL tool link to see the actual path (usually `/<project>/<job-name>/url-1`).
3. The model will be freed from GPU and reloaded by the server process.

**To add the tool:** RunAI UI > your workspace > Edit > Tools > Add Tool > Custom URL, port 8501

**To find the URL path:** RunAI UI > your workspace > Connect > click the Custom URL tool link. The path in the URL after the cluster host is your `STREAMLIT_BASE_PATH`.


In [ ]:
import subprocess

repo_dir = "/ocr/repo"

# Free notebook model from GPU before server loads its own
if 'model' in dir() and model is not None:
    del model
    del processor
    torch.cuda.empty_cache()
    print("Freed notebook model from GPU.")

# Start extraction server in local mode (loads model with transformers, no vLLM)
env = {**os.environ, "LLM_BASE_URL": "local", "HF_HUB_OFFLINE": "1"}
server_proc = subprocess.Popen(
    ["python", "ocr_app/scripts/ocr_server.py"],
    env=env, cwd=repo_dir,
)
print(f"Extraction server starting (PID {server_proc.pid})...")
print("  Mode: LOCAL (transformers, no vLLM)")
print(f"  Model: {MODEL_NAME}")
print("  All pages -> VLM image extraction")

# Wait for server to load model and be ready
import time as _time
for i in range(120):
    _time.sleep(2)
    try:
        import httpx
        resp = httpx.get("http://localhost:8090/health", timeout=2.0)
        if resp.status_code == 200:
            info = resp.json()
            print(f"\nServer ready! LLM: {info.get('llm_model', '?')}")
            break
    except Exception:
        if i % 15 == 14:
            print(f"  Still loading model... ({(i+1)*2}s)")
else:
    print("WARNING: Server did not become ready within 4 min")

# Start Streamlit UI with proxy-compatible base path
base_path = os.environ.get("STREAMLIT_BASE_PATH", "")
streamlit_cmd = [
    "streamlit", "run", "ocr_app/app.py",
    "--server.port=8501", "--server.address=0.0.0.0", "--server.headless=true",
]
if base_path:
    streamlit_cmd.append(f"--server.baseUrlPath={base_path}")

streamlit_proc = subprocess.Popen(
    streamlit_cmd,
    env={**os.environ, "OCR_SERVICE_URL": "http://localhost:8090"},
    cwd=repo_dir,
)
print(f"Streamlit started (PID {streamlit_proc.pid})")
if base_path:
    print(f"\nAccess: https://deepthought.doit.wisc.edu{base_path}/")
else:
    print("\nWARNING: STREAMLIT_BASE_PATH not set — proxy URL will 404")
    print("Add env var: STREAMLIT_BASE_PATH = /<project>/<job-name>/url-1")
print("\nTo stop: server_proc.terminate(); streamlit_proc.terminate()")
